In [1]:
!git clone https://github.com/AgentsLabAIy/DeepSeek.git

Cloning into 'DeepSeek'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 17 (delta 1), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 13.56 KiB | 1.36 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [20]:
%pwd

'/content/DeepSeek'

In [21]:
%ls

agent/  DeepSeek_r1.ipynb  requirements.txt


In [23]:
import sys

sys.path.insert(0, "/content/DeepSeek")

from agent.llm_invocation.invocation import invoke_llm

print("Agent modules imported successfully.")

Agent modules imported successfully.


In [24]:
%ls

agent/  DeepSeek_r1.ipynb  requirements.txt


In [12]:
%cd agent

/content/DeepSeek/agent


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log 2>&1 &

In [ ]:
import time
time.sleep(5)

In [ ]:
!curl http://localhost:11434/api/tags

{"models":[{"name":"deepseek-r1:7b","model":"deepseek-r1:7b","modified_at":"2026-11-01T16:32:42.855387726Z","size":4683075440,"digest":"755ced02ce7befdb13b7ca74e1e4d08cddba4986afdb63a480f2c93d3140383f","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3584,"runner":"ggml"},"capabilities":["tools","thinking","completion"]}]}

In [ ]:
!ollama pull deepseek-r1:7b

In [ ]:
!ollama run deepseek-r1:7b "What is the capital of Mexico?"

The capital of Mexico is Mexico City.



In [ ]:
!ollama pull deepseek-r1:70b

In [ ]:
!ollama run deepseek-r1:70b "Who is prime minister of India?"

^C


In [ ]:
!ollama ls

NAME              ID              SIZE      MODIFIED      
deepseek-r1:7b    755ced02ce7b    4.7 GB    2 minutes ago    


In [ ]:
!pip install -q fastapi uvicorn pyngrok requests

In [ ]:
import requests
import uvicorn
import threading

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "deepseek-r1:7b"

app = FastAPI(title="DeepSeek Colab Inference API")


class InferenceRequest(BaseModel):
    messages: list[dict]
    parameters: dict = Field(default_factory=dict)


@app.post("/generate")
def generate(request: InferenceRequest):

    if not request.messages:
        raise HTTPException(status_code=400, detail="Messages cannot be empty")

    payload = {
        "model": MODEL_NAME,
        "messages": request.messages,
        "stream": False,
        "options": request.parameters
    }

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json=payload,
            timeout=300
        )

        response.raise_for_status()
        result = response.json()

        return {
            "response": result["message"]["content"],
            "model": MODEL_NAME
        }

    except requests.exceptions.RequestException as exc:
        raise HTTPException(status_code=502, detail=str(exc))


@app.get("/health")
def health():

    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=10
        )
        response.raise_for_status()

        return {
            "status": "online",
            "ollama": "connected",
            "model": MODEL_NAME
        }

    except requests.exceptions.RequestException:
        raise HTTPException(status_code=503, detail="Ollama is unavailable")

In [ ]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

INFO:     Started server process [905]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

# Expose Colab port 8000
tunnel = ngrok.connect(8000, "http")

COLAB_API_URL = tunnel.public_url

print("Colab API URL:", COLAB_API_URL)
print("Health check:", COLAB_API_URL + "/health")

ERROR:pyngrok.process.ngrok:t=2026-11-01T17:35:56+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-11-01T17:35:56+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.